# APUSH FRQ Grader v3
Fresh Colab workflow for v3. It clones the repository, keeps run outputs on Drive, defaults to set1, and never opens set2 without an explicit locked-final flag.

## 1. Clone/update the repository and install dependencies

In [9]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/aryanjverma/slm.git'
REPO = Path('/content/apush-frq-grader-slm')
if (REPO / '.git').exists():
    subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=REPO, check=True)
elif REPO.exists():
    raise RuntimeError(f'{REPO} exists but is not a git checkout; remove or rename it first')
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
print(f'Repository: {REPO}\nCommit: {COMMIT}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[train]'], check=True)

def run_streaming(command, log_path=None):
    log_file = None
    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_file = log_path.open('a', encoding='utf-8')
        log_file.write(f'\nCOMMAND: {command}\n')
        log_file.flush()
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        if log_file is not None:
            log_file.write(line)
            log_file.flush()
    return_code = process.wait()
    if log_file is not None:
        log_file.close()
    if return_code:
        raise RuntimeError(f'Command failed with exit code {return_code}: {command}')


Repository: /content/apush-frq-grader-slm
Commit: 7076c3265476e80e5506b7c2b9085ff0fa707efc


## 2. Mount Drive and configure restart-safe outputs
After a disconnect, rerun Sections 1, 2, and 4, then rerun the interrupted training section. Only complete checkpoints are resumed; partial checkpoint directories are ignored.

In [10]:
from google.colab import drive
drive.mount('/content/drive')
RUN_ROOT = Path('/content/drive/MyDrive/slm_v3')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
V3_DATA = REPO / 'artifacts/data/v3/train_chat_v3.jsonl'

def complete_checkpoints(model_dir):
    model_dir = Path(model_dir)
    complete = []
    incomplete = []
    for checkpoint in model_dir.glob('checkpoint-*'):
        required = [checkpoint / 'trainer_state.json', checkpoint / 'adapter_config.json']
        has_weights = any(checkpoint.glob('adapter_model.*'))
        (complete if all(path.exists() for path in required) and has_weights else incomplete).append(checkpoint)
    key = lambda path: int(path.name.split('-')[-1])
    complete.sort(key=key)
    incomplete.sort(key=key)
    return complete, incomplete

def show_recovery_status(label, model_dir, eval_dir):
    complete, incomplete = complete_checkpoints(model_dir)
    final_ready = (Path(model_dir) / 'final' / 'adapter_config.json').exists()
    summaries = list(Path(eval_dir).glob('*_set1_summary.json'))
    print(f'{label}: final={final_ready}, latest_checkpoint={complete[-1] if complete else None}, summaries={len(summaries)}')
    if incomplete:
        print(f'Ignoring incomplete checkpoints: {incomplete}')

print(f'Run outputs: {RUN_ROOT}')
show_recovery_status('0.5B', RUN_ROOT / 'qwen-0.5b', RUN_ROOT / 'dev-0.5b')
show_recovery_status('1.5B', RUN_ROOT / 'qwen-1.5b', RUN_ROOT / 'dev-1.5b')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Run outputs: /content/drive/MyDrive/slm_v3
0.5B: final=True, latest_checkpoint=/content/drive/MyDrive/slm_v3/qwen-0.5b/checkpoint-39, summaries=3
1.5B: final=True, latest_checkpoint=/content/drive/MyDrive/slm_v3/qwen-1.5b/checkpoint-39, summaries=3


## 3. Reproduce the v2 failure analysis

In [30]:
subprocess.run([sys.executable, 'scripts/analyze_v2_for_v3.py'], check=True)


CompletedProcess(args=['/usr/bin/python3', 'scripts/analyze_v2_for_v3.py'], returncode=0)

## 4. Verify or rebuild the v3 dataset
The committed artifact is used by default. Rebuilding is explicit and will refuse to overwrite an existing immutable output directory.

In [3]:
import json

V3_MANIFEST = REPO / 'artifacts/data/v3/dataset_manifest_v3.json'
assert V3_DATA.exists() and V3_MANIFEST.exists(), 'Committed v3 dataset is missing'
manifest = json.loads(V3_MANIFEST.read_text(encoding='utf-8'))
assert manifest['rows'] == 200
print(json.dumps(manifest, indent=2))


{
  "files": {
    "audit": {
      "path": "artifacts/data/v3/dataset_audit_v3.json",
      "sha256": "88cb2473a5698275fde45e6b608f764ca879e6aa617bbd0bd416fd190cc0ae91"
    },
    "cases": {
      "path": "artifacts/data/v3/train_cases_v3.jsonl",
      "sha256": "5b4efcab413dee38dcb7c855f650c1a838663839c34ba89d7c59f7dc661a97fe"
    },
    "chat": {
      "path": "artifacts/data/v3/train_chat_v3.jsonl",
      "sha256": "c88f499b86b8bc12ba00cfe93c6a1d94afc96565765c397498bc8fe61dd1f63d"
    }
  },
  "rows": 200,
  "settings": {
    "min_long_essay_rate": 0.05,
    "seed": 13,
    "sources": [
      {
        "path": "artifacts/data/train_cases.jsonl",
        "sha256": "cd0d1e1c4aec00955cf1508edeefd49ff4170a3a52541ff5a979f13134832ef6"
      },
      {
        "path": "artifacts/data/v2/train_realistic_v2.jsonl",
        "sha256": "7ad33d7bf9e9fc49f05c3ac9e59861471dc96f6ba24ee633be06d950a17c6886"
      },
      {
        "path": "artifacts/data/v2/train_adversarial_v2.jsonl",
        "sha

## 5. Base 0.5B set1 benchmark

In [32]:
RUN_BASE_BENCHMARK = False  # The verified result is committed; set True to reproduce it.
BASE_EVAL_DIR = RUN_ROOT / 'base'
if RUN_BASE_BENCHMARK:
    subprocess.run([
        sys.executable, 'scripts/eval_v3.py',
        '--model', 'Qwen/Qwen2.5-0.5B-Instruct',
        '--model-name', 'Qwen2.5-0.5B-base',
        '--output-dir', str(BASE_EVAL_DIR),
    ], check=True)
    BASE_SUMMARY = sorted(BASE_EVAL_DIR.glob('*_set1_summary.json'))[-1]
else:
    BASE_SUMMARY = REPO / 'artifacts/eval/v3/base/04b2dada9fdb0cb3e27c_set1_summary.json'
assert BASE_SUMMARY.exists(), BASE_SUMMARY
print(BASE_SUMMARY)


/content/apush-frq-grader-slm/artifacts/eval/v3/base/04b2dada9fdb0cb3e27c_set1_summary.json


## 6. Assemble the three-configuration pretraining benchmark

In [33]:
BENCHMARK_PATH = RUN_ROOT / 'pretraining_dev_benchmark.json'
subprocess.run([
    sys.executable, 'scripts/benchmark_v3_dev.py',
    '--base-summary', str(BASE_SUMMARY),
    '--output', str(BENCHMARK_PATH),
], check=True)
print(BENCHMARK_PATH.read_text(encoding='utf-8'))


{
  "configurations": {
    "base_qwen_0_5b_layered": {
      "count": 27,
      "qwk": 0.2583,
      "repetition_rate": 0.0,
      "schema_valid_rate": 0.7778,
      "total_mae": 2.3704,
      "total_within_one_rate": 0.3704,
      "truncation_count": 1
    },
    "v2_raw_generation": {
      "count": 27,
      "qwk": 0.1825,
      "repetition_rate": 0.2593,
      "schema_valid_rate": 0.3333,
      "total_mae": 3.037,
      "total_within_one_rate": 0.2593,
      "truncation_count": 4
    },
    "v2_with_v3_structured_layer": {
      "count": 27,
      "qwk": 0.2222,
      "repetition_rate": 0.2593,
      "schema_valid_rate": 0.6667,
      "total_mae": 3.3333,
      "total_within_one_rate": 0.2222,
      "truncation_count": 4
    }
  },
  "note": "All three pretraining configurations are populated.",
  "rows": 27,
  "split": "set1"
}


## 7. Verify assistant-only labels and the 3,072-token context

In [34]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_v3_training.py'], check=True)
subprocess.run([
    sys.executable, 'scripts/audit_v3_tokens.py',
    '--output', str(RUN_ROOT / 'v3_training_token_audit.json'),
], check=True)


CompletedProcess(args=['/usr/bin/python3', 'scripts/audit_v3_tokens.py', '--output', '/content/drive/MyDrive/slm_v3/v3_training_token_audit.json'], returncode=0)

## 8. Train Qwen 0.5B
Training is opt-in. Each epoch checkpoint automatically runs set1 generation evaluation.

In [5]:
import torch

RUN_TRAINING = True  # Safe to rerun after a disconnect; completed work is detected on Drive.
FORCE_RETRAIN = False
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
MODEL05_DIR = RUN_ROOT / 'qwen-0.5b'
FINAL05 = MODEL05_DIR / 'final' / 'adapter_config.json'
if RUN_TRAINING and (FORCE_RETRAIN or not FINAL05.exists()):
    assert torch.cuda.is_available(), 'Select a GPU runtime before training'
    DEV05 = (
        f'{sys.executable} scripts/eval_v3.py --model {{checkpoint}} '
        f'--model-name Qwen2.5-0.5B-v3 --output-dir {RUN_ROOT / "dev-0.5b"}'
    )
    command = [
        sys.executable, 'scripts/train_v3.py',
        '--model', 'Qwen/Qwen2.5-0.5B-Instruct',
        '--output', str(MODEL05_DIR),
        '--data', str(V3_DATA),
        '--dev-eval-command', DEV05,
    ]
    if not USE_BF16:
        command.append('--no-bf16')
    prior_checkpoints, incomplete = complete_checkpoints(MODEL05_DIR)
    if prior_checkpoints:
        command += ['--resume-from-checkpoint', str(prior_checkpoints[-1])]
        print(f'Resuming from {prior_checkpoints[-1]}')
    run_streaming(command, MODEL05_DIR / 'colab_training.log')
elif FINAL05.exists():
    print(f'0.5B already complete: {FINAL05.parent}')
else:
    print('0.5B training skipped; set RUN_TRAINING=True to execute.')


0.5B already complete: /content/drive/MyDrive/slm_v3/qwen-0.5b/final


## 9. Train Qwen 1.5B with identical settings

In [6]:
MODEL15_DIR = RUN_ROOT / 'qwen-1.5b'
FINAL15 = MODEL15_DIR / 'final' / 'adapter_config.json'
if RUN_TRAINING and (FORCE_RETRAIN or not FINAL15.exists()):
    DEV15 = (
        f'{sys.executable} scripts/eval_v3.py --model {{checkpoint}} '
        f'--model-name Qwen2.5-1.5B-v3 --output-dir {RUN_ROOT / "dev-1.5b"}'
    )
    command = [
        sys.executable, 'scripts/train_v3.py',
        '--model', 'Qwen/Qwen2.5-1.5B-Instruct',
        '--output', str(MODEL15_DIR),
        '--data', str(V3_DATA),
        '--dev-eval-command', DEV15,
    ]
    if not USE_BF16:
        command.append('--no-bf16')
    prior_checkpoints, incomplete = complete_checkpoints(MODEL15_DIR)
    if prior_checkpoints:
        command += ['--resume-from-checkpoint', str(prior_checkpoints[-1])]
        print(f'Resuming from {prior_checkpoints[-1]}')
    run_streaming(command, MODEL15_DIR / 'colab_training.log')
elif FINAL15.exists():
    print(f'1.5B already complete: {FINAL15.parent}')
else:
    print('1.5B training skipped; set RUN_TRAINING=True to execute.')


1.5B already complete: /content/drive/MyDrive/slm_v3/qwen-1.5b/final


## 10. Collect checkpoint set1 summaries
The training callback evaluates every saved checkpoint; this cell verifies those summaries exist.

In [11]:
CHECKPOINT_SUMMARIES = sorted((RUN_ROOT / 'dev-0.5b').glob('*_set1_summary.json'))
CHECKPOINT_SUMMARIES += sorted((RUN_ROOT / 'dev-1.5b').glob('*_set1_summary.json'))
print(f'Checkpoint summaries: {len(CHECKPOINT_SUMMARIES)}')
for path in CHECKPOINT_SUMMARIES:
    print(path)


Checkpoint summaries: 6
/content/drive/MyDrive/slm_v3/dev-0.5b/2ac88dfc545f603d7966_set1_summary.json
/content/drive/MyDrive/slm_v3/dev-0.5b/a8a0048e21d6575db9c8_set1_summary.json
/content/drive/MyDrive/slm_v3/dev-0.5b/c50673f9c1c522ed073a_set1_summary.json
/content/drive/MyDrive/slm_v3/dev-1.5b/2798677196fd2c028463_set1_summary.json
/content/drive/MyDrive/slm_v3/dev-1.5b/c15c89617fe75aa6b6a5_set1_summary.json
/content/drive/MyDrive/slm_v3/dev-1.5b/f1938e7098e5ded10789_set1_summary.json


## 11. Rank passing checkpoints and freeze the candidate

In [12]:
LOCK_MANIFEST = RUN_ROOT / 'v3_final_lock.json'
RANKING_PATH = RUN_ROOT / 'checkpoint_ranking.json'
if CHECKPOINT_SUMMARIES:
    run_streaming([
        sys.executable, 'scripts/rank_v3_checkpoints.py',
        *map(str, CHECKPOINT_SUMMARIES),
        '--output', str(RANKING_PATH),
        '--lock-manifest', str(LOCK_MANIFEST),
    ], RUN_ROOT / 'checkpoint_ranking.log')
    print(RANKING_PATH.read_text(encoding='utf-8'))
else:
    print('No trained checkpoint summaries yet; no candidate was locked.')


No checkpoint passes set1 acceptance; set2 remains locked
Ranking written to /content/drive/MyDrive/slm_v3/checkpoint_ranking.json
{
  "ranking": [
    {
      "acceptance": {
        "mae_lte_1_5": false,
        "passed": false,
        "qwk_gte_0_35": false,
        "schema_valid_100pct": false,
        "within_one_gte_60pct": false,
        "zero_truncations": true
      },
      "identity": {
        "data_hash": "2cab5b8d0b2fba6b1a9d9213885b46a56af14ad5ce06c9b7e3bf9253e0856312",
        "decoding_settings": {
          "balanced_json_stopping": true,
          "do_sample": false,
          "max_new_tokens": 320,
          "score_enum_constraint": "scores_only_v2",
          "temperature": 0.0
        },
        "model_hash": "e34276462d68cf89ff0e71811418e7e280b24fef3927d8a282ffba5ea2986a9d",
        "model_name": "Qwen2.5-1.5B-v3",
        "prompt_version": "v3_scores_feedback_1",
        "run_id": "2798677196fd2c028463",
        "split": "set1"
      },
      "layered_system": {

## 12. Acceptance gate
Set2 remains closed unless layered set1 validity is 100%, truncations are zero, MAE is at most 1.5, within-one is at least 60%, and QWK is at least 0.35.

In [13]:
if LOCK_MANIFEST.exists():
    lock = json.loads(LOCK_MANIFEST.read_text(encoding='utf-8'))
    assert lock.get('set1_acceptance_passed') is True
    print('A passing set1 candidate is locked. Set2 is now eligible for one final run.')
else:
    print('No passing lock manifest. Set2 remains closed.')


No passing lock manifest. Set2 remains closed.


## 13. Official evaluation split lock
Section 13 defaults to set1 and does nothing until explicitly enabled. Set2 additionally requires `FINAL_EVALUATION=True` and the exact passing lock manifest.

In [14]:
RUN_OFFICIAL_EVAL = False
FINAL_EVALUATION = False  # False means set1; True requests the one locked set2 run.
LOCKED_MODEL = None  # Set to the selected checkpoint path from checkpoint_ranking.json.
LOCKED_MODEL_NAME = 'locked-v3-candidate'

if RUN_OFFICIAL_EVAL:
    assert LOCKED_MODEL, 'Set LOCKED_MODEL before evaluation'
    command = [
        sys.executable, 'scripts/eval_v3.py',
        '--model', str(LOCKED_MODEL),
        '--model-name', LOCKED_MODEL_NAME,
        '--output-dir', str(RUN_ROOT / 'official'),
    ]
    if FINAL_EVALUATION:
        assert LOCK_MANIFEST.exists(), 'Set2 requires a passing set1 lock manifest'
        command += ['--final-evaluation', '--lock-manifest', str(LOCK_MANIFEST)]
    subprocess.run(command, check=True)
else:
    print('Official evaluation disabled. Set2 has not been opened.')


Official evaluation disabled. Set2 has not been opened.
